<a href="https://colab.research.google.com/github/MaggieHDez/ClassFiles/blob/PLN/actividad7PLNMCHD255879.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
# !pip install unidecode spacy
# !python -m spacy download es_core_news_sm

In [76]:
import numpy as np
import pandas as pd
import spacy
from unidecode import unidecode
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

nlp = spacy.load("es_core_news_sm")
stopwords = nlp.Defaults.stop_words


# ---------- Corpus (crudo, SIN preprocesamiento) ----------
corpus_es = [
    "Me encantó la película, los actores fueron increíbles",
    "No me gustó la película, el guion fue muy malo",
    "La actuación fue excelente, pero la historia era predecible",
    "Película aburrida, me dormí a la mitad",
    "¡Maravillosa! Recomiendo esta película a todos mis amigos",
    "El guion estaba mal escrito, pero la actuación salvó la película",
    "Demasiado larga y lenta, no la volvería a ver",
    "Me gustó mucho, los efectos especiales fueron impresionantes",
    "No es mala, pero esperaba algo más emocionante",
    "Una obra maestra del cine español, increíble dirección",
    "El final fue confuso, no entendí nada",
    "Excelente cinematografía, pero el guion flojo",
    "Demasiado predecible, ya sabía lo que iba a pasar",
    "Me reí mucho, muy divertida y entretenida",
    "La música fue espectacular, pero los actores no convencieron",
    "Película mediocre, no aporta nada nuevo",
    "Excelente historia, emocionante hasta el final",
    "No me gustó la ambientación, parecía de bajo presupuesto",
    "Muy buena dirección y fotografía, la recomiendo",
    "El guion tenía agujeros, pero los efectos visuales impresionan",
    "Aburrida, diálogo poco natural",
    "Gran actuación de los protagonistas, realmente me impactó",
    "El ritmo es lento, pero la historia es interesante",
    "No la recomiendo, desperdicié mi tiempo",
    "Un clásico moderno, me encantó cada escena",
    "Las escenas de acción fueron espectaculares",
    "El humor es pobre y los personajes poco creíbles",
    "Me emocioné, la historia me llegó al corazón",
    "La trama es confusa y difícil de seguir",
    "Película fantástica, muy bien lograda"
]

In [77]:
# Funciones
def datos_matriz(X, nombre: str):
    n_docs, n_terms = X.shape
    print(f"\n[{nombre}]")
    print(f"  Dimensiones: {n_docs} documentos x {n_terms} términos")

def mostrar_primeros_tokens(vectorizer, k: int = 10):
    vocab = vectorizer.vocabulary_          # {token: índice}
    inv_vocab = sorted(vocab.items(), key=lambda x: x[1])[:k]
    print("\nPrimeros tokens e índices:")
    for tok, idx in inv_vocab:
        print(f"  idx={idx:>3}  token='{tok}'")

def comparar_matrices(Xa, Xb, nombre_a: str, nombre_b: str):
    nnz_a = Xa.nnz if hasattr(Xa, "nnz") else np.count_nonzero(Xa)
    nnz_b = Xb.nnz if hasattr(Xb, "nnz") else np.count_nonzero(Xb)
    dens_a = nnz_a / (Xa.shape[0] * Xa.shape[1])
    dens_b = nnz_b / (Xb.shape[0] * Xb.shape[1])
    print("\n Comparación general")
    print(f"- Dimensiones {nombre_a}: {Xa.shape} | {nombre_b}: {Xb.shape}")
    print(f"- Densidad    {nombre_a}: {dens_a:.4f} | {nombre_b}: {dens_b:.4f}")

# Aplicar documento por documento
def recorrer_corpus(corpus):
  corpus_preprocesado = []
  for i, doc in enumerate(corpus):
      texto = unidecode(doc.lower())
      tokens = texto.split()
      tokens_limpios = quitar_stopwords(tokens)
      corpus_preprocesado.append(" ".join(tokens_limpios))
  return corpus_preprocesado


# Eliminar stopwords
def quitar_stopwords(lemas):
  stop = nlp.Defaults.stop_words # Stopwords en Español
  limpios = [l for l in lemas if l not in stop and l.strip()]
  return limpios


# --------------------------------------------------
# Ejercicio 1 - Vectorización sin preprocesamiento
# --------------------------------------------------
print("\nEjercicio 1 - Vectorización sin preprocesamiento")
count_vec = CountVectorizer()
tfidf_vec = TfidfVectorizer()

X_count = count_vec.fit_transform(corpus_es)
X_tfidf  = tfidf_vec.fit_transform(corpus_es)

# Reportes
datos_matriz(X_count, "CountVectorizer - CRUDO")
mostrar_primeros_tokens(count_vec, k=10)

# Creamos el DataFrame BoW
bow_df = pd.DataFrame(X_count.toarray(), columns=count_vec.get_feature_names_out())
print("\n", bow_df.iloc[:30, :15].to_string(), "\n") # todas las filas y primeras 10 columnas

datos_matriz(X_tfidf, "TfidfVectorizer - CRUDO")
mostrar_primeros_tokens(tfidf_vec, k=10)

# Creamos el DataFrame TF-IDF
tfid_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out())
print("\n", tfid_df.iloc[:30, :15].to_string(), "\n") # todas las filas y primeras 10 columnas

comparar_matrices(X_count, X_tfidf, "Count (crudo)", "TF-IDF (crudo)")

# Diferencias de valores en matrices
print("\n¿Mismas dimensiones?", X_count.shape == X_tfidf.shape)
print("¿Matrices idénticas en valores?", np.allclose(X_count.toarray(), X_tfidf.toarray()))

# Inspección de SOLO un documento
doc_id = 0
terms = count_vec.get_feature_names_out()

df_count = (pd.DataFrame({"term": terms, "count": X_count[doc_id].toarray().ravel()})
            .query("count > 0").sort_values("count", ascending=False))
df_tfidf = (pd.DataFrame({"term": terms, "tfidf": X_tfidf[doc_id].toarray().ravel()})
            .query("tfidf > 0").sort_values("tfidf", ascending=False))

print("\n Documento 0 (CountVectorizer)")
print(df_count.to_string(index=False))

print("\n Documento 0 (TfidfVectorizer)")
print(df_tfidf.to_string(index=False))

# --------------------------------------------------
# Ejercicio 2 - Vectorización con preprocesamiento
# --------------------------------------------------

print("\nEjercicio 2 - Vectorización con preprocesamiento")
count_vec_prep = CountVectorizer(
    lowercase=True,
    strip_accents='unicode'
)

tfidf_vec_prep = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode'
)

corpus_preprocesado = recorrer_corpus(corpus_es)

X_count_prep = count_vec_prep.fit_transform(corpus_preprocesado)
X_tfidf_prep  = tfidf_vec_prep.fit_transform(corpus_preprocesado)

# Reportes
datos_matriz(X_count_prep, "CountVectorizer - PREPROCESADO")
mostrar_primeros_tokens(count_vec_prep, k=10)

# Creamos el DataFrame BoW
bow_prep_df = pd.DataFrame(X_count_prep.toarray(), columns=count_vec_prep.get_feature_names_out())
print("\n", bow_prep_df.iloc[:30, :15].to_string(), "\n") # todas las filas y primeras 10 columnas

datos_matriz(X_tfidf_prep, "TfidfVectorizer - PREPROCESADO")
mostrar_primeros_tokens(tfidf_vec_prep, k=10)

# Creamos el DataFrame TF-IDF
tfid_prep_df = pd.DataFrame(X_tfidf_prep.toarray(), columns=tfidf_vec_prep.get_feature_names_out())
print("\n", tfid_prep_df.iloc[:30, :15].to_string(), "\n") # todas las filas y primeras 10 columnas

comparar_matrices(X_count_prep, X_tfidf_prep, "Count (preprocesado)", "TF-IDF (preprocesado)")

# Diferencias de valores en matrices
print("\n¿Mismas dimensiones?", X_count_prep.shape == X_tfidf_prep.shape)
print("¿Matrices idénticas en valores?", np.allclose(X_count_prep.toarray(), X_tfidf_prep.toarray()))

# Inspección de SOLO un documento
doc_id = 0
terms = count_vec_prep.get_feature_names_out()

df_count_prep = (pd.DataFrame({"term": terms, "count": X_count_prep[doc_id].toarray().ravel()})
            .query("count > 0").sort_values("count", ascending=False))
df_tfidf_prep = (pd.DataFrame({"term": terms, "tfidf": X_tfidf_prep[doc_id].toarray().ravel()})
            .query("tfidf > 0").sort_values("tfidf", ascending=False))

print("\n Documento 0 (CountVectorizer)")
print(df_count_prep.to_string(index=False))

print("\n Documento 0 (TfidfVectorizer)")
print(df_tfidf_prep.to_string(index=False))


Ejercicio 1 - Vectorización sin preprocesamiento

[CountVectorizer - CRUDO]
  Dimensiones: 30 documentos x 127 términos

Primeros tokens e índices:
  idx=  0  token='aburrida'
  idx=  1  token='acción'
  idx=  2  token='actores'
  idx=  3  token='actuación'
  idx=  4  token='agujeros'
  idx=  5  token='al'
  idx=  6  token='algo'
  idx=  7  token='ambientación'
  idx=  8  token='amigos'
  idx=  9  token='aporta'

     aburrida  acción  actores  actuación  agujeros  al  algo  ambientación  amigos  aporta  bajo  bien  buena  cada  cine
0          0       0        1          0         0   0     0             0       0       0     0     0      0     0     0
1          0       0        0          0         0   0     0             0       0       0     0     0      0     0     0
2          0       0        0          1         0   0     0             0       0       0     0     0      0     0     0
3          1       0        0          0         0   0     0             0       0       0   